# Combiner RAG et affinage : RAFT et encodeur adapté sur la presse de 1914

Le RAG du chapitre 4 répond aux questions sur les 64 598 passages de la presse de 1914 : recherche hybride (BM25 et recherche dense fusionnées par RRF), puis une réponse sourcée, ou un refus, au format `Reponse` (`trouve`, `reponse`, `citations`). Ses mesures ont désigné deux faiblesses : la recherche dense rate des passages que BM25 trouve (l'encodeur), et le petit modèle qui répond donne des réponses que les passages servis ne soutiennent pas, ou cite mal (le générateur). Ce TP entraîne les deux composants, chacun pour sa panne, puis mesure ce que chacun apporte, sur le jeu de test du chapitre 4, avant de recommander une configuration.

1. Construire le jeu RAFT : des questions servies avec les passages de la recherche, l'oracle parmi des distracteurs, ou absent
2. Affiner le générateur par QLoRA sur ce jeu
3. Adapter l'encodeur sur des paires question–passage, réindexer, mesurer la recherche
4. Répondre dans les cinq configurations du cours
5. Comparer : le tableau, la marge, question par question
6. Rédiger la recommandation

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, sur lequel il tient en une heure et demie à deux heures (une estimation), dont une heure de calcul : un quart d'heure pour affiner Qwen3-1.7B, autant pour réindexer le corpus, une demi-heure pour répondre et juger. Sans GPU, il tourne quand même, en trois quarts d'heure environ sur deux processeurs comme ceux de Colab, en plus petit : Qwen3-0.6B en float32 au lieu de Qwen3-1.7B en 4 bits, deux pas d'entraînement pour le générateur et un pour l'encodeur (le code tourne, les adaptateurs n'apprennent rien), quatre passages réencodés au lieu de 64 598. À la place des vôtres, les exercices 3 et 4 chargent alors l'encodeur adapté, son index et le générateur que ce notebook a entraînés sur GPU ; l'exercice 4 ne répond qu'à 3 questions, et les exercices 5 et 6 lisent les réponses et les notes que ce même code a produites sur GPU, sur le jeu entier.

## Données et modèles

Les passages et leurs *embeddings* (512 dimensions, l'index de l'encodeur de base), les 20 questions du chapitre 2 et les questions générées et validées du chapitre 4 (le jeu de test), et `training-questions.jsonl` : 619 questions générées pour l'entraînement, sur d'autres passages que ceux du jeu de test, chacune avec sa réponse courte, la citation du passage qui la porte (`quote`) et ce passage (`passage_id`). Puis, pour le chemin sans GPU, ce que ce notebook a produit sur GPU : l'adaptateur de l'encodeur et l'index qu'il a réencodé, l'adaptateur RAFT de Qwen3-0.6B, et l'évaluation des cinq configurations. Ollama s'installe et télécharge le juge en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/questions.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/synthetic-questions.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/training-questions.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/lab-results/raft-encoder-embeddings-512.npy
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/lab-results/raft-evaluation.jsonl
!wget -q -P press-encoder-lora-qwen3-embedding-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/press-encoder-lora-qwen3-embedding-0.6b/adapter_model.safetensors
!wget -q -P press-encoder-lora-qwen3-embedding-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/press-encoder-lora-qwen3-embedding-0.6b/adapter_config.json
!wget -q -P press-raft-lora-qwen3-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/press-raft-lora-qwen3-0.6b/adapter_model.safetensors
!wget -q -P press-raft-lora-qwen3-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/press-raft-lora-qwen3-0.6b/adapter_config.json
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bm25s ragas peft trl bitsandbytes
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# Le générateur, celui qu'on affine : sur GPU, Qwen3-1.7B ; sans GPU, Qwen3-0.6B.
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
# Le juge du chapitre 4, servi par Ollama : sur GPU, Ministral 3 8B (Mistral AI), plus
# gros et d'une autre famille ; sans GPU, Qwen3-1.7B.
MODELE_JUGE = "ministral-3:8b" if GPU else "qwen3:1.7b"
print("GPU" if GPU else "Sans GPU", ":", MODELE, "répond,", MODELE_JUGE, "juge")
# Le juge ne sert qu'à l'exercice 4 : Ollama démarre et le télécharge pendant les
# entraînements. 8 192 jetons de contexte : le prompt de fidélité de Ragas dépasse les
# 4 096 par défaut.
env = {
  **os.environ,
  "OLLAMA_NUM_PARALLEL": "4" if GPU else "1",
  "OLLAMA_CONTEXT_LENGTH": "8192",
}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
telechargement = subprocess.Popen(
  f"sleep 5 && ollama pull {MODELE_JUGE}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

## Le service du chapitre 4

Le RAG à améliorer, donné. `classer(question, configuration, index)` renvoie le classement des passages (leurs positions dans `passages`) : `"dense"`, la recherche dense seule, ou `"hybride"`, BM25 et dense fusionnés par RRF, la recherche du service ; `index` nomme l'index dense, `"base"` (l'encodeur Qwen3-Embedding-0.6B et `embeddings-512.npy`), puis, à l'exercice 3, `"adapte"`. Chaque classement est calculé une fois, puis gardé. `rechercher(question, k, configuration, index)` en garde les `k` premiers.

Le service sert à son générateur la consigne du chapitre 4, `INSTRUCTIONS`, et les passages numérotés avec leur journal et leur date (`messages_service(question, servis)`). Au chapitre 4, Ollama imposait le format `Reponse` par une grammaire ; ici, le générateur tourne dans transformers, qui ne le fait pas, et la consigne du service, `CONSIGNE`, ajoute une ligne qui donne ce format. `lire_reponse(texte)` lit la sortie du générateur : un objet `Reponse`, ou `None` si elle n'en contient pas de lisible, que la mesure compte comme un refus.

Le jeu de test, `jeu`, est celui du chapitre 4 : les 20 questions du chapitre 2 (`origine` `"main"`) et les 43 questions générées que l'expert a validées (`"generee"`), avec leurs passages de référence (`passage_ids`). `PASSAGES_TEST` réunit les passages que nomment ses deux fichiers, questions rejetées comprises : aucun ne doit entrer dans l'entraînement. Lancez la cellule (une à deux minutes, durée indicative).

In [ ]:
# bm25s se sert de JAX s'il est installé (c'est le cas sur Colab), et JAX réserve
# d'office 75 % de la mémoire du GPU : on le cantonne au CPU, le GPU est aux modèles.
os.environ["JAX_PLATFORMS"] = "cpu"

import asyncio
import gc
import hashlib
import json
import re
import time
from collections import defaultdict
from functools import cache

import bm25s
import numpy as np
import pandas as pd
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if GPU else "cpu"
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16. Sans
# GPU, float32 : les demi-précisions y sont très lentes.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32

passages = pd.read_parquet("passages.parquet")
par_id = passages.set_index("id")
position = {id_passage: i for i, id_passage in enumerate(passages.id)}
vecteurs = np.load("embeddings-512.npy").astype(np.float32)

ENCODEUR = "Qwen/Qwen3-Embedding-0.6B"
encodeur = SentenceTransformer(
  ENCODEUR, device=DEVICE, truncate_dim=512, model_kwargs={"dtype": DTYPE}
)
encodeur.max_seq_length = 512  # comme pour l'index
index_mots = bm25s.BM25()
index_mots.index(
  bm25s.tokenize(passages.text.tolist(), show_progress=False), show_progress=False
)
# Les index denses : l'encodeur des requêtes et les vecteurs des passages, qui vont
# ensemble. L'exercice 3 ajoute "adapte".
INDEX = {"base": (encodeur, vecteurs)}


def rrf(*classements, k: int = 60) -> list[int]:
  scores = defaultdict(float)
  for classement in classements:
    for rang, doc in enumerate(classement, start=1):
      scores[doc] += 1 / (k + rang)
  return sorted(scores, key=scores.get, reverse=True)


@cache
def classer_dense(question: str, index: str = "base") -> list[int]:
  modele, vecteurs_index = INDEX[index]
  requete = modele.encode_query(question).astype(np.float32)
  requete /= np.linalg.norm(requete)
  return np.argsort(-(vecteurs_index @ requete))[:100].tolist()


@cache
def classer(question: str, configuration: str = "hybride", index: str = "base"):
  dense = classer_dense(question, index)
  if configuration == "dense":
    return dense
  lexicale = index_mots.retrieve(
    bm25s.tokenize([question], show_progress=False), k=100, show_progress=False
  )[0][0]
  return rrf(lexicale.tolist(), dense)


def rechercher(
  question: str, k: int = 5, configuration: str = "hybride", index: str = "base"
) -> list[int]:
  return classer(question, configuration, index)[:k]


class Reponse(BaseModel):
  trouve: bool
  reponse: str
  citations: list[int]


INSTRUCTIONS = """Tu réponds aux questions sur la presse française de 1914, uniquement à partir
des passages fournis, numérotés. Réponds en français, en une ou deux phrases, et mets dans
citations les numéros des passages qui justifient ta réponse (pas dans la réponse elle-même).
Si les passages ne contiennent pas la réponse, mets trouve à false et n'invente rien."""
FORMAT = """Réponds par un objet JSON : {"trouve": true ou false, "reponse": "...", "citations": [numéros des passages]}."""
CONSIGNE = INSTRUCTIONS + "\n" + FORMAT


def contexte(i: int) -> str:
  return f"({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"


def messages_service(question: str, servis: list[int]) -> list[dict]:
  """La conversation que le service envoie au générateur : la consigne, puis les
  passages servis, numérotés à partir de 1, et la question."""
  if servis:
    textes = "\n\n".join(f"[{n}] {contexte(i)}" for n, i in enumerate(servis, start=1))
  else:
    textes = "(aucun passage)"
  return [
    {"role": "system", "content": CONSIGNE},
    {"role": "user", "content": f"Passages :\n{textes}\n\nQuestion : {question}"},
  ]


def lire_reponse(texte: str) -> Reponse | None:
  """Le premier objet JSON de la sortie, s'il a le format Reponse ; None sinon."""
  try:
    objet, _ = json.JSONDecoder().raw_decode(texte[texte.find("{") :])
    return Reponse.model_validate(objet)
  except (ValueError, ValidationError):
    return None


def mots_pleins(texte: str) -> set[str]:
  """Les mots de 4 lettres ou plus d'un texte, en minuscules (chapitre 4)."""
  return set(re.findall(r"\w{4,}", texte.lower()))


questions = pd.read_json("questions.jsonl", lines=True)
# Une question sans réponse a `answer` à null : pandas en fait un NaN, que `or` prend
# pour une valeur. On le remplace une fois pour toutes par None.
questions["answer"] = questions.answer.astype(object).where(
  questions.answer.notna(), None
)
generees = pd.read_json("synthetic-questions.jsonl", lines=True)
jeu = pd.concat(
  [
    questions.assign(origine="main"),
    generees[generees.valid].assign(origine="generee"),
  ],
  ignore_index=True,
)[["id", "question", "answer", "passage_ids", "origine"]]
PASSAGES_TEST = (
  set(questions.passage_ids.explode().dropna())
  | set(generees.passage_ids.explode())
  | set(generees.source_passage_id)
)
print(len(jeu), "questions de test,", len(PASSAGES_TEST), "passages de test")
for i in rechercher("Où Jaurès a-t-il été assassiné ?", 3):
  print(passages.id[i], passages.text[i][:150])

## Les mesures du chapitre 4

Données aussi, telles que le chapitre 4 les a écrites :

- `mesurer(ligne, configuration, index)` : pour une question du jeu qui a une réponse, la `reussite` à 5 (un passage de référence parmi les 5 premiers) et `rr` (`1 / rang` du premier passage de référence) ;
- `verifier(ligne)` : pour une réponse, les mesures par script. `refus_juste` vaut 1 si le générateur refuse alors qu'aucun passage de référence n'est servi (une question sans réponse, une recherche qui rate son passage, ou aucun passage) ; `refus_a_tort`, 1 s'il refuse alors qu'un passage de référence est servi ; `citation_juste`, 1 si une réponse donnée cite un passage de référence. Chacune vaut `NaN` là où elle ne s'applique pas ;
- `noter(ligne)` : les notes du juge, par Ragas (`Faithfulness`, la fidélité aux passages servis, et `AnswerAccuracy`, l'exactitude face à la référence) ;
- `marge(p, n)` : la marge d'un taux `p` sur `n` questions, et `comparer(avant, apres)` : les questions gagnées et perdues entre deux séries de succès.

Une réponse est juste (`juste`) si le juge lui donne une exactitude d'au moins 0,5, ou si elle refuse une question sans réponse. Le juge a la même place qu'au chapitre 4, avec ses limites : il ne vaut que contrôlé (exercice 5 du chapitre 4).

In [ ]:
import instructor
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerAccuracy, Faithfulness


def mesurer(ligne, configuration: str, index: str = "base", k: int = 5) -> dict:
  classement = passages.id[classer(ligne.question, configuration, index)].tolist()
  pertinents = set(ligne.passage_ids)
  rangs = [r for r, i in enumerate(classement, start=1) if i in pertinents]
  return {
    "reussite": bool(pertinents & set(classement[:k])),
    "rr": 1 / rangs[0] if rangs else 0.0,
  }


def verifier(ligne) -> dict:
  appui = bool(set(ligne.servis) & set(ligne.passage_ids))
  return {
    "refus_juste": np.nan if appui else float(not ligne.trouve),
    "refus_a_tort": float(not ligne.trouve) if appui else np.nan,
    "citation_juste": (
      float(bool(set(ligne.sources) & set(ligne.passage_ids)))
      if ligne.trouve and ligne.answer is not None
      else np.nan
    ),
  }


llm = llm_factory(
  MODELE_JUGE,
  client=AsyncOpenAI(
    base_url="http://localhost:11434/v1", api_key="ollama", timeout=3600
  ),
  mode=instructor.Mode.JSON,
  temperature=0,
  max_tokens=1024,
  reasoning_effort="none",
)
fidelite = Faithfulness(llm=llm)
exactitude = AnswerAccuracy(llm=llm)


async def juger(mesure, **entrees) -> float:
  try:
    return (await mesure.ascore(**entrees)).value
  except Exception as erreur:
    print("Le juge a échoué :", erreur)
    return np.nan


async def noter(ligne) -> dict:
  if not ligne.trouve:
    # Un refus n'affirme rien, et ne donne pas la réponse attendue : pas de juge.
    return {
      "fidelite": np.nan,
      "exactitude": 0.0 if ligne.answer is not None else np.nan,
    }
  notes = {"fidelite": np.nan, "exactitude": np.nan}
  if ligne.servis:  # sans passages servis, rien à quoi être fidèle
    notes["fidelite"] = await juger(
      fidelite,
      user_input=ligne.question,
      response=ligne.reponse,
      retrieved_contexts=[contexte(position[i]) for i in ligne.servis],
    )
  if ligne.answer is not None:
    notes["exactitude"] = await juger(
      exactitude,
      user_input=ligne.question,
      response=ligne.reponse,
      reference=ligne.answer,
    )
  return notes


def marge(p: float, n: int) -> float:
  return 1.96 * np.sqrt(p * (1 - p) / n)


def comparer(avant: pd.Series, apres: pd.Series) -> tuple[list, list]:
  communes = avant.index.intersection(apres.index)
  gagnees = [i for i in communes if not avant[i] and apres[i]]
  perdues = [i for i in communes if avant[i] and not apres[i]]
  return gagnees, perdues


def memoire_go(modele) -> float:
  """La mémoire qu'occupe le modèle, en Go : celle du GPU, sinon celle de ses poids."""
  if GPU:
    return torch.cuda.memory_allocated() / 1e9
  return modele.get_memory_footprint() / 1e9


def pic_go() -> float:
  """Le pic de mémoire GPU depuis le dernier libere(), en Go (0 sans GPU)."""
  return torch.cuda.max_memory_allocated() / 1e9 if GPU else 0.0


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`, et remet le pic à zéro."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 1. Construire le jeu RAFT

RAFT affine le générateur dans les conditions du service : chaque exemple est une question servie avec $k$ passages, comme en service, et la réponse attendue. Écrivez `exemple_raft(ligne, rng)` : pour une question d'entraînement (une ligne de `a_construire`), elle renvoie un dictionnaire avec `prompt`, la conversation du service (`messages_service`, mot pour mot), `completion`, la réponse cible, un message `assistant` au format `Reponse` en JSON, et, pour les compter, `id`, `split`, `oracle` (l'oracle est-il servi ?), `rang` (sa position, à partir de 0, ou `None`) et `ecartes` (les faux distracteurs écartés).

- **Les distracteurs** : les premiers passages que la recherche du service (`classer(ligne.question)`, hybride, index de base) classe pour la question, sauf l'oracle (`ligne.passage_id`), sauf les passages de test (`PASSAGES_TEST`), et sauf les faux distracteurs, ceux qui donnent la réponse : écrivez d'abord `contient_reponse(texte, ligne)`, vraie si le passage contient la citation de l'oracle (`ligne.quote`, espaces normalisés : une reprise du même texte), ou au moins la moitié des mots pleins de la réponse qui ne sont pas dans la question (ceux de la question sont dans tous les passages bien classés).
- **L'oracle** est servi dans une part `P_ORACLE` des exemples (`rng.random() < P_ORACLE`), avec `K - 1` distracteurs, à une position tirée au hasard (`rng.integers(K)`) : la cible est la réponse (`ligne.answer`), qui cite ce passage. Sinon, `K` distracteurs, et la cible est un refus : `trouve` à faux, la réponse `REFUS`, aucune citation. Le cours l'explique : sans oracle, RAFT garde la réponse pour cible, et le modèle l'apprend par cœur ; l'assistant de la presse doit refuser hors du corpus.

Construisez `raft`, un `DataFrame` d'un exemple par question de `a_construire`. Combien d'exemples ont leur oracle ? L'oracle est-il réparti sur les cinq positions ? Combien de faux distracteurs la règle a-t-elle écartés : lisez-en quelques-uns, donnent-ils vraiment la réponse ? Vérifiez aussi la règle sur l'oracle lui-même : `contient_reponse` doit y être vraie pour toutes les questions. Mesurez enfin la longueur des exemples en tokens (le prompt et la cible), contre la longueur maximale de l'entraînement, `max_length` à l'exercice 2 (3 072 tokens).

La cible ne porte que les trois champs du service. L'article de RAFT fait écrire au modèle un raisonnement qui cite le passage utile avant la réponse (`##Reason: … ##begin_quote## … ##end_quote## … ##Answer: …`), et `quote` en donnerait la citation. On s'en passe ici : le format `Reponse` n'a pas de champ pour lui, en ajouter un changerait le service avec le modèle (deux changements dans une même ligne du tableau), et chaque réponse en serait plus longue, donc plus lente. Une variante à essayer après le TP : un champ `extrait` avant `reponse`, puis la même comparaison.

Sans GPU, `a_construire` n'a que 16 questions, 8 de chaque `split` : la recherche encode chaque question en deux secondes environ sur deux processeurs, et l'entraînement de l'exercice 2 n'en prend que 3.

**Documentation utile**

- [`Generator.random`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html)
- [`Generator.integers`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.integers.html)
- [`BaseModel.model_dump`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump)
- [`apply_chat_template`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.apply_chat_template)

In [ ]:
from transformers import AutoTokenizer

entrainement_q = pd.read_json("training-questions.jsonl", lines=True)
print(entrainement_q.split.value_counts().to_dict())
# Sans GPU, 8 questions de chaque split (moins d'une minute, durée indicative).
a_construire = entrainement_q if GPU else entrainement_q.groupby("split").head(8)
K = 5  # les passages servis, comme en service
P_ORACLE = 0.8  # la part des exemples où l'oracle est servi
REFUS = "Les passages fournis ne contiennent pas la réponse."
rng = np.random.default_rng(1914)
tokenizer = AutoTokenizer.from_pretrained(MODELE)

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `" ".join(texte.split())` : le texte, ses espaces normalisés (l'OCR en met parfois deux) ; `a in b` cherche la sous-chaîne `a` dans `b`
- `mots_pleins(ligne.answer) - mots_pleins(ligne.question)` : les mots pleins de la réponse absents de la question ; `len(mots & mots_pleins(texte))`, ceux que le passage contient
- `position[ligne.passage_id]` : la position de l'oracle dans `passages` ; `passages.id[i]`, `passages.text[i]` : l'identifiant et le texte du passage à la position `i`
- `liste.insert(rang, oracle)` place l'oracle parmi les distracteurs ; il est alors le passage numéro `rang + 1`
- la cible : `Reponse(trouve=True, reponse=ligne.answer, citations=[rang + 1])`, puis `json.dumps(cible.model_dump(), ensure_ascii=False)`, le JSON que le générateur doit écrire, accents compris
- `pd.DataFrame([exemple_raft(ligne, rng) for ligne in a_construire.itertuples()])`
- les longueurs : `tokenizer.apply_chat_template(e.prompt + e.completion, enable_thinking=False)["input_ids"]`, les tokens de l'exemple, sous la forme que lui donnera `trl`
- un exemple à vérifier à la main : sur une ligne dont `quote` vaut `"payé environ 400 fr. le mille"`, `contient_reponse` est vraie pour un passage qui dit `"est  payé environ 400 fr. le mille, livré"`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def contient_reponse(texte: str, ligne) -> bool:
  if " ".join(ligne.quote.split()) in " ".join(texte.split()):
    return True
  mots = mots_pleins(ligne.answer) - ...
  return bool(mots) and len(mots & mots_pleins(texte)) >= ...


def exemple_raft(ligne, rng) -> dict:
  oracle = position[ligne.passage_id]
  distracteurs, ecartes = [], []
  for i in classer(ligne.question):
    if i == oracle or passages.id[i] in ...:
      continue
    if contient_reponse(passages.text[i], ligne):
      ecartes.append(i)
    else:
      distracteurs.append(i)
    if len(distracteurs) == K:
      break
  avec_oracle = bool(rng.random() < P_ORACLE)
  if avec_oracle:
    rang = int(rng.integers(K))
    servis = distracteurs[: K - 1]
    servis.insert(rang, oracle)
    cible = Reponse(trouve=True, reponse=..., citations=[...])
  else:
    rang = None
    servis = distracteurs[:K]
    cible = Reponse(trouve=..., reponse=REFUS, citations=[])
  return {
    "id": ligne.id,
    "split": ligne.split,
    "oracle": avec_oracle,
    "rang": rang,
    "ecartes": ecartes,
    "prompt": messages_service(ligne.question, ...),
    "completion": [
      {"role": "assistant", "content": json.dumps(..., ensure_ascii=False)}
    ],
  }


raft = pd.DataFrame([exemple_raft(ligne, rng) for ligne in a_construire.itertuples()])
print(
  raft.groupby("split").size().to_dict(), f"oracle servi : {raft.oracle.mean():.0%}"
)
print(
  "Positions de l'oracle :",
  raft.rang.dropna().astype(int).value_counts().sort_index().to_dict(),
)
print(f"{raft.ecartes.str.len().sum()} faux distracteurs écartés")
lignes = a_construire.set_index("id")
for exemple in raft[raft.ecartes.str.len() > 0].head(2).itertuples():
  ligne = lignes.loc[exemple.id]
  print(f"\nQ : {ligne.question}\nR : {ligne.answer}")
  print("  écarté :", passages.text[exemple.ecartes[0]][:300])
oracles = [
  contient_reponse(par_id.text[ligne.passage_id], ligne)
  for ligne in a_construire.itertuples()
]
print(f"\nLa règle reconnaît l'oracle de {np.mean(oracles):.0%} des questions")
longueurs = pd.Series(
  [
    len(
      tokenizer.apply_chat_template(e.prompt + e.completion, enable_thinking=...)[
        "input_ids"
      ]
    )
    for e in raft.itertuples()
  ]
)
print(
  f"De {longueurs.min()} à {longueurs.max()} tokens, {longueurs.mean():.0f} en moyenne"
)
print(raft.completion[0][0]["content"])
```

### Solution

In [ ]:
def contient_reponse(texte: str, ligne) -> bool:
  if " ".join(ligne.quote.split()) in " ".join(texte.split()):
    return True
  mots = mots_pleins(ligne.answer) - mots_pleins(ligne.question)
  return bool(mots) and len(mots & mots_pleins(texte)) >= len(mots) / 2


def exemple_raft(ligne, rng) -> dict:
  oracle = position[ligne.passage_id]
  distracteurs, ecartes = [], []
  for i in classer(ligne.question):
    if i == oracle or passages.id[i] in PASSAGES_TEST:
      continue
    if contient_reponse(passages.text[i], ligne):
      ecartes.append(i)
    else:
      distracteurs.append(i)
    if len(distracteurs) == K:
      break
  avec_oracle = bool(rng.random() < P_ORACLE)
  if avec_oracle:
    rang = int(rng.integers(K))
    servis = distracteurs[: K - 1]
    servis.insert(rang, oracle)
    cible = Reponse(trouve=True, reponse=ligne.answer, citations=[rang + 1])
  else:
    rang = None
    servis = distracteurs[:K]
    cible = Reponse(trouve=False, reponse=REFUS, citations=[])
  return {
    "id": ligne.id,
    "split": ligne.split,
    "oracle": avec_oracle,
    "rang": rang,
    "ecartes": ecartes,
    "prompt": messages_service(ligne.question, servis),
    "completion": [
      {
        "role": "assistant",
        "content": json.dumps(cible.model_dump(), ensure_ascii=False),
      }
    ],
  }


raft = pd.DataFrame([exemple_raft(ligne, rng) for ligne in a_construire.itertuples()])
print(
  raft.groupby("split").size().to_dict(), f"oracle servi : {raft.oracle.mean():.0%}"
)
print(
  "Positions de l'oracle :",
  raft.rang.dropna().astype(int).value_counts().sort_index().to_dict(),
)
print(f"{raft.ecartes.str.len().sum()} faux distracteurs écartés")
lignes = a_construire.set_index("id")
for exemple in raft[raft.ecartes.str.len() > 0].head(2).itertuples():
  ligne = lignes.loc[exemple.id]
  print(f"\nQ : {ligne.question}\nR : {ligne.answer}")
  print("  écarté :", passages.text[exemple.ecartes[0]][:300])
oracles = [
  contient_reponse(par_id.text[ligne.passage_id], ligne)
  for ligne in a_construire.itertuples()
]
print(f"\nLa règle reconnaît l'oracle de {np.mean(oracles):.0%} des questions")
longueurs = pd.Series(
  [
    len(
      tokenizer.apply_chat_template(e.prompt + e.completion, enable_thinking=False)[
        "input_ids"
      ]
    )
    for e in raft.itertuples()
  ]
)
print(
  f"De {longueurs.min()} à {longueurs.max()} tokens, {longueurs.mean():.0f} en moyenne"
)
print(raft.completion[0][0]["content"])

# Lors d'une exécution sans GPU, sur 16 questions : l'oracle est servi dans 12 exemples,
# à chacune des cinq positions. Sur les 619 (le même code, hors de ce notebook) : 83 %,
# chaque position presque autant ; 608 faux distracteurs écartés, dans 37 % des
# exemples ; 1 640 tokens en moyenne, un peu plus de 3 000 au plus. La règle des faux
# distracteurs est prudente : elle écarte aussi des passages qui ne répondent pas
# (un autre journal de Berlin, une autre peine de prison). Cela ne coûte rien, le
# passage suivant prend la place ; garder un passage qui répond apprendrait au modèle à
# ignorer une bonne réponse. Elle reconnaît l'oracle de chaque question : la citation
# est recopiée du passage. La cible ne fait qu'une trentaine de tokens.

## 2. Affiner le générateur

La cellule ci-dessous charge le générateur de base, `base` (sur GPU, Qwen3-1.7B en NF4 avec double quantification ; sans GPU, Qwen3-0.6B en float32), la configuration LoRA du chapitre 5, `qlora` (rang 16, toutes les couches linéaires), les réglages d'entraînement, `REFERENCE`, et fait de `raft` les deux jeux de `trl`, `entrainement` et `validation` : un exemple par ligne, `prompt` et `completion`, avec `chat_template_kwargs` qui coupe la réflexion de Qwen3 (`enable_thinking=False`), comme au service.

Écrivez `prepare()`, qui renvoie un `SFTTrainer` prêt à entraîner `qlora` sur `base` pendant une époque d'`entrainement`, avec les réglages de `REFERENCE`, `validation` pour l'évaluer, et les journaux tous les 5 pas, sans TensorBoard ni checkpoint : l'adaptateur s'enregistre à la fin. Sur un jeu *prompt / completion*, `trl` ne compte dans la perte que la complétion (`completion_only_loss`, actif par défaut sur ce format) : regardez, dans le premier exemple préparé (`trainer.train_dataset[0]`), les tokens qui comptent. Que reste-t-il des passages et de la question ? Le plus long exemple tient-il dans `max_length` ?

Mesurez la perte de validation avant l'entraînement (`trainer.evaluate()`), entraînez, mesurez-la après, et enregistrez l'adaptateur dans `raft-lora` (`trainer.save_model`). Rangez dans `bilan` la durée de l'entraînement en minutes, le pic de mémoire (`pic_go()`), les deux pertes de validation et la taille de l'adaptateur en Mo. De combien la perte baisse-t-elle ? Dans la cible, qu'est-ce qui s'apprend vite, et qu'est-ce qui demande de lire les passages ?

Sur un T4, comptez un quart d'heure environ (une estimation : 557 exemples de 1 600 tokens en moyenne). Sans GPU, `entrainement` n'a que 2 exemples et `validation` 1 : deux pas d'un exemple, pour voir la boucle tourner, cinq à dix minutes.

**Documentation utile**

- [`SFTConfig`](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig)
- [La perte sur la complétion seule](https://huggingface.co/docs/trl/sft_trainer#train-on-completion-only)
- [`Trainer.evaluate`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.evaluate)
- [`Trainer.save_model`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.save_model)

In [ ]:
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

NF4 = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=DTYPE,
)
# Sans GPU, en float32 : bitsandbytes est lent sur processeur.
base = AutoModelForCausalLM.from_pretrained(
  MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=NF4 if GPU else None
)
print(f"Générateur de base : {memoire_go(base):.2f} Go")
qlora = LoraConfig(
  r=16,
  lora_alpha=32,
  lora_dropout=0.05,
  target_modules="all-linear",
  task_type="CAUSAL_LM",
)
# Exemples par pas et pas accumulés par mise à jour : 8 exemples sur GPU, 1 sans. Un
# exemple par pas : les plus longs font 3 000 tokens.
LOT = 1
ACCUMULATION = 8 if GPU else 1
REFERENCE = dict(
  per_device_train_batch_size=LOT,
  per_device_eval_batch_size=LOT,
  gradient_accumulation_steps=ACCUMULATION,
  num_train_epochs=1,
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  optim="paged_adamw_8bit" if GPU else "adamw_torch",
  # Moins de mémoire au prix d'un passage avant de plus ; sans GPU aussi : les
  # activations de 1 600 tokens y pèseraient plusieurs Go.
  gradient_checkpointing=True,
  bf16=BF16,
  fp16=GPU and not BF16,
  # Le plus long exemple fait un peu plus de 3 000 tokens : plus court, trl couperait sa
  # fin, la cible.
  max_length=3072,
)


def jeu_trl(split: str) -> Dataset:
  exemples = raft[raft.split == split]
  return Dataset.from_list(
    [
      {
        "prompt": e.prompt,
        "completion": e.completion,
        "chat_template_kwargs": {"enable_thinking": False},
      }
      for e in exemples.itertuples()
    ]
  )


entrainement, validation = jeu_trl("train"), jeu_trl("validation")
if not GPU:
  entrainement, validation = entrainement.select(range(2)), validation.select(range(1))
print(len(entrainement), "exemples d'entraînement,", len(validation), "de validation")
libere()

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- dans les réglages : `logging_steps=5` ; `report_to="none"` ; `save_strategy="no"` : pas de checkpoint, l'adaptateur s'enregistre à la fin, par `save_model` ; l'évaluation se lance à la main, avant et après
- `SFTConfig(output_dir="raft", **(REFERENCE | reglages))` : `|` réunit deux dictionnaires
- `SFTTrainer(model=base, args=config, train_dataset=entrainement, eval_dataset=validation, processing_class=tokenizer, peft_config=qlora)`
- un exemple préparé a des `input_ids` et des `labels` de même longueur : `-100` dans `labels` exclut le token de la perte ; `tokenizer.decode(tokens)` les remet en texte
- `trainer.evaluate()["eval_loss"]` : la perte de validation ; `trainer.save_model("raft-lora")` enregistre l'adaptateur seul, `adapter_model.safetensors` et `adapter_config.json`
- `os.path.getsize(os.path.join("raft-lora", "adapter_model.safetensors"))` : sa taille en octets

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def prepare() -> SFTTrainer:
  reglages = dict(logging_steps=5, report_to=..., save_strategy="no")
  config = SFTConfig(output_dir="raft", **(REFERENCE | reglages))
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=...,
    eval_dataset=...,
    processing_class=tokenizer,
    peft_config=qlora,
  )


trainer = prepare()
exemple = trainer.train_dataset[0]
comptes = [t for t, l in zip(exemple["input_ids"], exemple["labels"]) if l != ...]
print(f"{len(comptes)} tokens sur {len(exemple['input_ids'])} comptent dans la perte :")
print(repr(tokenizer.decode(comptes)))
longueurs = [len(ids) for ids in trainer.train_dataset["input_ids"]]
print(
  f"Le plus long exemple : {max(longueurs)} tokens ; max_length",
  REFERENCE["max_length"],
)

avant = trainer.evaluate()["eval_loss"]
debut = time.time()
trainer.train()
minutes = (time.time() - debut) / 60
apres = ...
trainer.save_model("raft-lora")
bilan = {
  "minutes": round(minutes, 1),
  "pic_go": round(pic_go(), 1),
  "perte_avant": round(avant, 3),
  "perte_apres": round(apres, 3),
  "adaptateur_mo": round(os.path.getsize(...) / 1e6, 1),
}
print(bilan)
```

### Solution

In [ ]:
def prepare() -> SFTTrainer:
  reglages = dict(logging_steps=5, report_to="none", save_strategy="no")
  config = SFTConfig(output_dir="raft", **(REFERENCE | reglages))
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=entrainement,
    eval_dataset=validation,
    processing_class=tokenizer,
    peft_config=qlora,
  )


trainer = prepare()
exemple = trainer.train_dataset[0]
comptes = [t for t, l in zip(exemple["input_ids"], exemple["labels"]) if l != -100]
print(f"{len(comptes)} tokens sur {len(exemple['input_ids'])} comptent dans la perte :")
print(repr(tokenizer.decode(comptes)))
longueurs = [len(ids) for ids in trainer.train_dataset["input_ids"]]
print(
  f"Le plus long exemple : {max(longueurs)} tokens ; max_length",
  REFERENCE["max_length"],
)

avant = trainer.evaluate()["eval_loss"]
debut = time.time()
trainer.train()
minutes = (time.time() - debut) / 60
apres = trainer.evaluate()["eval_loss"]
trainer.save_model("raft-lora")
bilan = {
  "minutes": round(minutes, 1),
  "pic_go": round(pic_go(), 1),
  "perte_avant": round(avant, 3),
  "perte_apres": round(apres, 3),
  "adaptateur_mo": round(
    os.path.getsize(os.path.join("raft-lora", "adapter_model.safetensors")) / 1e6, 1
  ),
}
print(bilan)

# Seule la cible compte dans la perte : le JSON de la réponse et la fin du tour, une
# trentaine de tokens sur 1 600 ; la consigne, les passages et la question n'y comptent
# pas. Ce que le modèle doit apprendre tient en peu de tokens : `trouve`, le numéro du
# passage cité, les premiers mots de la réponse ; le reste du format s'apprend en
# quelques pas. Lors d'une exécution sans GPU, deux pas sur deux exemples font déjà
# baisser la perte de l'unique exemple de validation (de 1,48 à 1,11) : la forme de la
# cible, pas encore la lecture. L'adaptateur pèse quelques dizaines de Mo.

## 3. Adapter l'encodeur

La cellule ci-dessous rend la mémoire du générateur et prépare les paires de l'encodeur : `paires`, les questions d'entraînement du split `train`, chacune avec son passage (`ligne.passage_id`) ; leurs classements par l'encodeur de base sont déjà calculés pour celles de l'exercice 1.

Écrivez `negatif(ligne)` : le négatif difficile d'une paire, la position du premier passage que l'encodeur de base lui-même classe haut (`classer(ligne.question, "dense")`), à partir du rang `RANG_MIN` (les premiers rangs sont les plus exposés aux faux négatifs), qui n'est ni son passage, ni un passage de test, ni un faux négatif (`contient_reponse`, de l'exercice 1) ; `None` s'il n'y en a pas. C'est ce que fait `mine_hard_negatives` de sentence-transformers (son `range_min`), mais lui réencode le corpus qu'on lui donne : l'index de base contient déjà les vecteurs des 64 598 passages, et les encoder une fois de plus prendrait un quart d'heure de T4.

Construisez ensuite `triplets`, un `Dataset` aux colonnes `anchor` (la question), `positive` (le texte de son passage) et `negative` (celui du négatif), puis adaptez l'encodeur : `encodeur_adapte`, l'encodeur de base rechargé en float32 et doté d'un adaptateur LoRA (`add_adapter`, la configuration `lora_encodeur` ci-dessous), entraîné une époque par `SentenceTransformerTrainer` avec `MultipleNegativesRankingLoss` (chaque question contre son passage, son négatif et les passages des autres paires du lot), `batch_sampler="no_duplicates"`, et la consigne des requêtes du modèle sur les questions (`prompts`). Un adaptateur plutôt que tout l'encodeur : l'entraînement tient sur le T4, et la version à garder pèse 40 Mo au lieu de 1,2 Go, l'encodeur de base restant partagé. Enregistrez-le dans `encodeur-lora`.

Puis réindexez : encodez les passages de `a_reencoder` avec `encodeur_adapte` (`encode_document`, normalisés), dans `vecteurs_adaptes`, et estimez la durée de la réindexation complète, `minutes_reindexation`. L'index et son encodeur forment une version : on ne mélange jamais des requêtes encodées par l'un et des passages encodés par l'autre.

Sur un T4, l'entraînement prend quelques minutes et la réindexation un quart d'heure environ (une estimation). Sans GPU, `paires` n'a que 2 questions, un pas, et `a_reencoder` 4 passages (cinq minutes environ) : l'estimation dit ce que coûterait la réindexation sur deux processeurs.

**Documentation utile**

- [`SentenceTransformer.add_adapter`](https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html#sentence_transformers.sentence_transformer.model.SentenceTransformer.add_adapter)
- [`SentenceTransformerTrainer`](https://sbert.net/docs/package_reference/sentence_transformer/trainer.html#sentence_transformers.sentence_transformer.trainer.SentenceTransformerTrainer)
- [`SentenceTransformerTrainingArguments`](https://sbert.net/docs/package_reference/sentence_transformer/training_args.html)
- [`MultipleNegativesRankingLoss`](https://sbert.net/docs/package_reference/sentence_transformer/losses.html#multiplenegativesrankingloss)
- [`BatchSamplers`](https://sbert.net/docs/package_reference/sentence_transformer/sampler.html#sentence_transformers.training_args.BatchSamplers)
- [`mine_hard_negatives`](https://sbert.net/docs/package_reference/util/hard_negatives.html#sentence_transformers.util.hard_negatives.mine_hard_negatives)
- [`SentenceTransformer.encode_document`](https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html#sentence_transformers.sentence_transformer.model.SentenceTransformer.encode_document)

In [ ]:
from sentence_transformers import (
  SentenceTransformerTrainer,
  SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss

# La mémoire du générateur rendue : l'exercice 4 le recharge.
del trainer, base
libere()

paires = entrainement_q[entrainement_q.split == "train"]
# Sans GPU : 2 paires, dont la recherche de l'exercice 1 a déjà classé les questions.
if not GPU:
  paires = paires[paires.id.isin(raft.id)].head(2)
RANG_MIN = 10
# Les paires par lot : 32 sur GPU, et autant de passages qu'en a le lot comme négatifs.
LOT_ENCODEUR = 32 if GPU else 2
lora_encodeur = LoraConfig(
  r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear"
)
# Les passages à réencoder : tous sur GPU ; sans GPU, 4.
a_reencoder = passages.index if GPU else passages.index[:4]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `classer(ligne.question, "dense")[RANG_MIN:]` : le classement dense, sans ses `RANG_MIN` premiers passages
- `Dataset.from_list([{"anchor": ..., "positive": ..., "negative": ...}, ...])` : une ligne par paire qui a un négatif ; `par_id.text[ligne.passage_id]`, le texte de son passage
- `SentenceTransformer(ENCODEUR, device=DEVICE, truncate_dim=512, model_kwargs={"dtype": torch.float32})`, puis `max_seq_length = 512` et `add_adapter(lora_encodeur)` : en float32 même sur GPU, où le calcul passe en demi-précision (`fp16`) mais où les poids entraînés doivent rester en float32
- dans `SentenceTransformerTrainingArguments(output_dir="encodeur", ...)` : `num_train_epochs=1`, `per_device_train_batch_size=LOT_ENCODEUR`, `learning_rate=1e-4` (un adaptateur apprend avec un taux plus fort que l'encodeur entier, `2e-5` dans le cours), `warmup_steps=0.1`, `batch_sampler="no_duplicates"`, `prompts={"anchor": encodeur_adapte.prompts["query"]}` (la consigne que `encode_query` met devant chaque requête), `gradient_checkpointing=True`, `fp16=GPU and not BF16`, `bf16=BF16`, `report_to="none"`, `save_strategy="no"`
- `SentenceTransformerTrainer(model=encodeur_adapte, args=args, train_dataset=triplets, loss=MultipleNegativesRankingLoss(encodeur_adapte)).train()`
- `encodeur_adapte.save_pretrained("encodeur-lora")` ; après l'entraînement, `encodeur_adapte.to(DTYPE)` le passe en float16 sur GPU pour encoder plus vite, comme l'index de base l'a été
- `encode_document(textes, batch_size=64, normalize_embeddings=True)` ; `astype(np.float16)`, la précision de l'index ; `time.time()` avant et après donne la durée, et `len(passages) / len(a_reencoder)` l'extrapole

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def negatif(ligne) -> int | None:
  for i in classer(ligne.question, "dense")[RANG_MIN:]:
    if (
      passages.id[i] != ligne.passage_id
      and passages.id[i] not in PASSAGES_TEST
      and not contient_reponse(passages.text[i], ...)
    ):
      return i
  return None


triplets = Dataset.from_list(
  [
    {"anchor": ligne.question, "positive": ..., "negative": passages.text[n]}
    for ligne in paires.itertuples()
    if (n := negatif(ligne)) is not None
  ]
)
print(len(triplets), "triplets")

encodeur_adapte = SentenceTransformer(
  ENCODEUR, device=DEVICE, truncate_dim=512, model_kwargs={"dtype": torch.float32}
)
encodeur_adapte.max_seq_length = 512
encodeur_adapte.add_adapter(...)
args = SentenceTransformerTrainingArguments(
  output_dir="encodeur",
  num_train_epochs=1,
  per_device_train_batch_size=LOT_ENCODEUR,
  learning_rate=1e-4,
  warmup_steps=0.1,
  batch_sampler=...,
  prompts={"anchor": ...},
  gradient_checkpointing=True,
  fp16=GPU and not BF16,
  bf16=BF16,
  report_to="none",
  save_strategy="no",
)
debut = time.time()
SentenceTransformerTrainer(
  model=encodeur_adapte,
  args=args,
  train_dataset=triplets,
  loss=MultipleNegativesRankingLoss(...),
).train()
minutes_encodeur = (time.time() - debut) / 60
encodeur_adapte.save_pretrained("encodeur-lora")
encodeur_adapte.to(DTYPE)

debut = time.time()
vecteurs_adaptes = encodeur_adapte.encode_document(
  passages.text[a_reencoder].tolist(), batch_size=64, normalize_embeddings=True
).astype(np.float16)
minutes_reindexation = (time.time() - debut) / 60 * len(passages) / ...
print(
  f"Entraînement : {minutes_encodeur:.1f} min ; réindexation complète : "
  f"{minutes_reindexation:.0f} min"
)
```

### Solution

In [ ]:
def negatif(ligne) -> int | None:
  for i in classer(ligne.question, "dense")[RANG_MIN:]:
    if (
      passages.id[i] != ligne.passage_id
      and passages.id[i] not in PASSAGES_TEST
      and not contient_reponse(passages.text[i], ligne)
    ):
      return i
  return None


triplets = Dataset.from_list(
  [
    {
      "anchor": ligne.question,
      "positive": par_id.text[ligne.passage_id],
      "negative": passages.text[n],
    }
    for ligne in paires.itertuples()
    if (n := negatif(ligne)) is not None
  ]
)
print(len(triplets), "triplets")

encodeur_adapte = SentenceTransformer(
  ENCODEUR, device=DEVICE, truncate_dim=512, model_kwargs={"dtype": torch.float32}
)
encodeur_adapte.max_seq_length = 512
encodeur_adapte.add_adapter(lora_encodeur)
args = SentenceTransformerTrainingArguments(
  output_dir="encodeur",
  num_train_epochs=1,
  per_device_train_batch_size=LOT_ENCODEUR,
  learning_rate=1e-4,
  warmup_steps=0.1,
  batch_sampler="no_duplicates",
  prompts={"anchor": encodeur_adapte.prompts["query"]},
  gradient_checkpointing=True,
  fp16=GPU and not BF16,
  bf16=BF16,
  report_to="none",
  save_strategy="no",
)
debut = time.time()
SentenceTransformerTrainer(
  model=encodeur_adapte,
  args=args,
  train_dataset=triplets,
  loss=MultipleNegativesRankingLoss(encodeur_adapte),
).train()
minutes_encodeur = (time.time() - debut) / 60
encodeur_adapte.save_pretrained("encodeur-lora")
encodeur_adapte.to(DTYPE)

debut = time.time()
vecteurs_adaptes = encodeur_adapte.encode_document(
  passages.text[a_reencoder].tolist(), batch_size=64, normalize_embeddings=True
).astype(np.float16)
minutes_reindexation = (time.time() - debut) / 60 * len(passages) / len(a_reencoder)
print(
  f"Entraînement : {minutes_encodeur:.1f} min ; réindexation complète : "
  f"{minutes_reindexation:.0f} min"
)

# Le négatif d'une paire est un passage que l'encodeur de base croit proche de la
# question sans qu'il y réponde : la perte apprend à l'en éloigner, comme des passages
# des autres paires du lot. Lors d'une exécution sans GPU, réencoder 4 passages laisse
# prévoir neuf jours de calcul pour les 64 598 : un encodeur adapté demande un GPU à chaque
# réindexation, et l'index réencodé ne vaut qu'avec lui.

### Mesurer la recherche

La cellule ci-dessous enregistre l'index adapté sous `"adapte"`. Sans GPU, votre encodeur n'a fait qu'un pas et n'a réencodé que 4 passages : elle charge à leur place l'adaptateur que ce notebook a entraîné sur GPU et l'index qu'il a réencodé.

Mesurez la recherche avant et après, sur les questions du jeu qui ont une réponse (`avec_reponse`) : construisez `recherche`, une ligne par index (`"base"`, `"adapte"`), par configuration (`"dense"`, `"hybride"`) et par question, avec l'index, la configuration, l'`id`, l'`origine` et les mesures de `mesurer`, puis affichez la réussite à 5 et le MRR moyens par configuration, par index et par origine. L'encodeur adapté retrouve-t-il mieux les passages en recherche dense ? Le gain survit-il à la fusion avec BM25 ? Est-il le même pour les questions du chapitre 2, écrites à la main, et pour les générées, du même style que celles de l'entraînement ? Comparez enfin, question par question (`comparer`), la recherche hybride avant et après.

Sans GPU, comptez quatre minutes : l'encodeur adapté encode les 61 questions, une à une.

In [ ]:
if not GPU:
  del encodeur_adapte  # le vôtre, d'un pas : sa mémoire rendue avant de charger l'autre
  libere()
  encodeur_adapte = SentenceTransformer(
    ENCODEUR, device=DEVICE, truncate_dim=512, model_kwargs={"dtype": DTYPE}
  )
  encodeur_adapte.max_seq_length = 512
  encodeur_adapte.load_adapter("press-encoder-lora-qwen3-embedding-0.6b")
  vecteurs_adaptes = np.load("raft-encoder-embeddings-512.npy")
INDEX["adapte"] = (encodeur_adapte, vecteurs_adaptes.astype(np.float32))
avec_reponse = jeu[jeu.passage_ids.str.len() > 0]
print(len(avec_reponse), "questions avec une réponse ;", vecteurs_adaptes.shape)

# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `{"index": index, "configuration": configuration, "id": ligne.id, "origine": ligne.origine, **mesurer(ligne, configuration, index)}` : une ligne de `recherche`, pour chaque `index`, chaque `configuration` et chaque `ligne` de `avec_reponse.itertuples()`
- `recherche.groupby(["configuration", "index", "origine"]).agg(n=("id", "size"), reussite=("reussite", "mean"), mrr=("rr", "mean"))`
- `recherche.pivot_table(index="id", columns="index", values="reussite")` sur les lignes de la configuration hybride : une colonne de succès par index, que `comparer` reçoit

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
recherche = pd.DataFrame(
  [
    {
      "index": index,
      "configuration": configuration,
      "id": ligne.id,
      "origine": ligne.origine,
      **mesurer(ligne, configuration, ...),
    }
    for index in ("base", "adapte")
    for configuration in ("dense", "hybride")
    for ligne in avec_reponse.itertuples()
  ]
)
print(
  recherche.groupby(["configuration", "index", "origine"])
  .agg(n=("id", "size"), reussite=(..., "mean"), mrr=("rr", "mean"))
  .round(2)
)
hybride = recherche[recherche.configuration == "hybride"].pivot_table(
  index="id", columns="index", values="reussite"
)
gagnees, perdues = comparer(hybride[...], hybride["adapte"])
print(f"Recherche hybride, gagnées : {gagnees}, perdues : {perdues}")
```

#### Solution

In [ ]:
recherche = pd.DataFrame(
  [
    {
      "index": index,
      "configuration": configuration,
      "id": ligne.id,
      "origine": ligne.origine,
      **mesurer(ligne, configuration, index),
    }
    for index in ("base", "adapte")
    for configuration in ("dense", "hybride")
    for ligne in avec_reponse.itertuples()
  ]
)
print(
  recherche.groupby(["configuration", "index", "origine"])
  .agg(n=("id", "size"), reussite=("reussite", "mean"), mrr=("rr", "mean"))
  .round(2)
)
hybride = recherche[recherche.configuration == "hybride"].pivot_table(
  index="id", columns="index", values="reussite"
)
gagnees, perdues = comparer(hybride["base"], hybride["adapte"])
print(f"Recherche hybride, gagnées : {gagnees}, perdues : {perdues}")

# La mesure se lit en deux temps : en recherche dense, où l'encodeur seul décide, puis
# en recherche hybride, celle du service, où BM25 rattrape déjà une part de ses échecs :
# un gain en dense peut fondre après la fusion. Par origine : un gain sur les seules
# questions générées, du même générateur que celles de l'entraînement, dirait que
# l'encodeur a appris leur style plutôt que la presse de 1914. Les questions gagnées et
# perdues disent si le gain vaut l'index à refaire.

## 4. Répondre dans les cinq configurations

Le tableau du cours compare cinq configurations, un composant changeant d'une ligne à l'autre : le générateur de base sans passages, puis avec ceux de la recherche du service (le RAG du chapitre 4), le générateur affiné sans passages, puis avec, puis avec l'encodeur adapté. Sans passages, le service envoie la même conversation, `(aucun passage)` à la place des passages : un générateur qui répond quand même répond de mémoire.

La cellule ci-dessous rend la mémoire des encodeurs (les classements des questions du jeu sont gardés), et donne `generer(modele, conversations)`, qui renvoie la sortie du modèle pour chaque conversation, sans échantillonnage et sans réflexion, avec la durée moyenne par réponse, et `charge_generateur(adaptateur)`, le générateur de base en `DTYPE`, sans quantification, et, si on lui nomme un adaptateur, cet adaptateur fusionné dans ses poids, comme on le sert. `GENERATEURS` nomme les deux générateurs, et `CONFIGURATIONS` les cinq configurations : le générateur et l'index de la recherche (`None` : sans passages).

Écrivez la boucle qui produit `reponses` : pour chaque générateur, chargé une fois, et chacune de ses configurations, les réponses aux questions de `a_comparer`, une ligne par question, avec la configuration, l'`id`, l'`origine`, la `question`, l'`answer` et les `passage_ids` de la question, puis `trouve`, `reponse`, `servis` (les identifiants des passages servis), `sources` (ceux que la réponse cite) et `secondes` (la durée par réponse). Une sortie illisible (`lire_reponse` renvoie `None`) compte pour un refus, et `lisible` le note. La cellule suivante vérifie les réponses et les fait noter par le juge.

Sur un T4, comptez cinq minutes pour les 315 réponses, par lots de 8 (une estimation). Sans GPU, `a_comparer` n'a que 3 questions, et le générateur affiné est l'adaptateur RAFT de Qwen3-0.6B que ce notebook a entraîné sur GPU : dix minutes environ, une réponse avec passages prenant près d'une minute.

**Documentation utile**

- [`PeftModel.from_pretrained`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.from_pretrained)
- [`merge_and_unload`](https://huggingface.co/docs/peft/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.merge_and_unload)

In [ ]:
from transformers import AutoModelForCausalLM

# Les classements des questions du jeu gardés en mémoire (ceux des questions qui ont une
# réponse le sont depuis l'exercice 3), les encodeurs ne servent plus.
for question in jeu.question:
  for index in INDEX:
    classer_dense(question, index)
INDEX.clear()
del encodeur, encodeur_adapte
libere()
tokenizer.padding_side = "left"  # pour générer par lots
TAILLE_LOT = 8 if GPU else 1


def generer(modele, conversations: list[list[dict]]) -> tuple[list[str], float]:
  """La sortie de `modele` pour chaque conversation, par lots de TAILLE_LOT, sans
  échantillonnage, et la durée moyenne par réponse, en secondes."""
  textes, debut = [], time.time()
  for i in range(0, len(conversations), TAILLE_LOT):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[i : i + TAILLE_LOT]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=150,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes, (time.time() - debut) / len(conversations)


def charge_generateur(adaptateur: str | None = None):
  modele = AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)
  if adaptateur:
    modele = PeftModel.from_pretrained(modele, adaptateur).merge_and_unload()
  return modele.eval()


# Sans GPU, le générateur affiné est celui que ce notebook a entraîné sur GPU, de
# Qwen3-0.6B.
GENERATEURS = {
  "base": None,
  "affine": "raft-lora" if GPU else "press-raft-lora-qwen3-0.6b",
}
CONFIGURATIONS = {
  "base, sans passages": ("base", None),
  "RAG + base": ("base", "base"),
  "affiné, sans passages": ("affine", None),
  "RAG + affiné": ("affine", "base"),
  "RAG + encodeur adapté + affiné": ("affine", "adapte"),
}
# Sans GPU, 3 questions : la recherche sert le passage de référence de la question 1,
# rate celui de la 4, et la 19 n'a pas de réponse dans le corpus.
a_comparer = jeu if GPU else jeu[jeu.id.isin([1, 4, 19])]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `for generateur, adaptateur in GENERATEURS.items()` : `modele = charge_generateur(adaptateur)`, puis ses configurations, celles dont le générateur est `generateur` ; `del modele` et `libere()` avant le suivant
- `rechercher(ligne.question, 5, "hybride", index) if index else []` : les passages servis, leurs positions dans `passages` ; `passages.id[servis].tolist()`, leurs identifiants
- `generer(modele, [messages_service(...) for ...])` : les sorties de toutes les questions d'une configuration d'un coup, et la durée par réponse
- `ligne._asdict()` : les colonnes d'une question de `a_comparer` en dictionnaire, avec `Index`, sa position, à retirer (`del`)
- `lue = lire_reponse(texte)` ; `lue.citations` donne des numéros à partir de 1 : `ids[n - 1]` pour ceux qui désignent un passage servi (`1 <= n <= len(ids)`)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
lignes = []
for generateur, adaptateur in GENERATEURS.items():
  modele = charge_generateur(...)
  for nom, (de, index) in CONFIGURATIONS.items():
    if de != generateur:
      continue
    servis = [
      rechercher(ligne.question, 5, "hybride", index) if index else []
      for ligne in a_comparer.itertuples()
    ]
    textes, secondes = generer(
      modele,
      [messages_service(q, s) for q, s in zip(a_comparer.question, ...)],
    )
    for ligne, positions, texte in zip(a_comparer.itertuples(), servis, textes):
      lue = lire_reponse(texte)
      ids = passages.id[positions].tolist()
      colonnes = ligne._asdict()  # les colonnes de la question
      del colonnes["Index"]
      lignes.append(
        {
          "configuration": nom,
          **colonnes,
          "lisible": lue is not None,
          "trouve": lue is not None and lue.trouve,
          "reponse": lue.reponse if lue else texte,
          "servis": ids,
          "sources": [ids[n - 1] for n in lue.citations if ...] if lue else [],
          "secondes": secondes,
        }
      )
    print(f"{nom} : {secondes:.1f} s par réponse")
  del modele
  libere()
reponses = pd.DataFrame(lignes)
print(
  reponses.groupby("configuration", sort=False)[["lisible", "trouve"]].mean().round(2)
)
```

### Solution

In [ ]:
lignes = []
for generateur, adaptateur in GENERATEURS.items():
  modele = charge_generateur(adaptateur)
  for nom, (de, index) in CONFIGURATIONS.items():
    if de != generateur:
      continue
    servis = [
      rechercher(ligne.question, 5, "hybride", index) if index else []
      for ligne in a_comparer.itertuples()
    ]
    textes, secondes = generer(
      modele,
      [messages_service(q, s) for q, s in zip(a_comparer.question, servis)],
    )
    for ligne, positions, texte in zip(a_comparer.itertuples(), servis, textes):
      lue = lire_reponse(texte)
      ids = passages.id[positions].tolist()
      colonnes = ligne._asdict()  # les colonnes de la question
      del colonnes["Index"]
      lignes.append(
        {
          "configuration": nom,
          **colonnes,
          "lisible": lue is not None,
          "trouve": lue is not None and lue.trouve,
          "reponse": lue.reponse if lue else texte,
          "servis": ids,
          "sources": (
            [ids[n - 1] for n in lue.citations if 1 <= n <= len(ids)] if lue else []
          ),
          "secondes": secondes,
        }
      )
    print(f"{nom} : {secondes:.1f} s par réponse")
  del modele
  libere()
reponses = pd.DataFrame(lignes)
print(
  reponses.groupby("configuration", sort=False)[["lisible", "trouve"]].mean().round(2)
)

# Lors d'une exécution sans GPU, Qwen3-0.6B de base refuse la question 1 alors que la
# recherche sert le passage qui y répond, et répond à la 4 sans passage qui le dise
# (« Le juge a arrêté Villain ») : les deux pannes que vise RAFT. Une réponse avec
# passages prend près d'une minute, sans passages une douzaine de secondes : le prompt fait 1 600 tokens
# ou une centaine. Fusionné, l'adaptateur ne coûte rien de plus que le modèle de base.

### Vérifier et noter

La cellule ci-dessous vérifie toutes les réponses par script (`verifier`), puis fait noter celles de `a_noter` par le juge (`noter`) : sur GPU, toutes, quatre appels au juge par réponse qui n'est pas un refus, une demi-heure environ (une estimation) ; sans GPU, les deux réponses avec passages à la question 4, du générateur de base et de l'affiné, cinq minutes environ. Elle enregistre l'évaluation dans `evaluation.jsonl`, une ligne par réponse. Lancez-la.

In [ ]:
telechargement.wait()  # le juge est téléchargé
verifications = pd.DataFrame(
  [verifier(ligne) for ligne in reponses.itertuples()], index=reponses.index
)
a_noter = (
  reponses
  if GPU
  else reponses[
    (reponses.id == 4) & reponses.configuration.isin(["RAG + base", "RAG + affiné"])
  ]
)
notes = await asyncio.gather(*(noter(ligne) for ligne in a_noter.itertuples()))
evaluation = reponses.join(verifications).join(pd.DataFrame(notes, index=a_noter.index))
juste = (evaluation.exactitude >= 0.5) | (evaluation.answer.isna() & ~evaluation.trouve)
# Juste ou non : seulement pour les réponses notées.
evaluation["juste"] = juste.astype(float).where(evaluation.index.isin(a_noter.index))
evaluation.to_json("evaluation.jsonl", orient="records", lines=True, force_ascii=False)
print(
  evaluation.loc[a_noter.index, ["configuration", "id", "trouve", "reponse"]]
  .join(evaluation[["fidelite", "exactitude"]])
  .to_string(max_colwidth=60)
)

## 5. Comparer

Sans GPU, vos réponses ne portent que sur 3 questions, et un générateur de Qwen3-0.6B : la cellule ci-dessous remplace `evaluation` par celle que ce même code a produite sur GPU, avec Qwen3-1.7B et ses adaptateurs, sur les 63 questions du jeu, notées par Ministral 3 8B.

Remplissez le tableau du cours : `tableau`, une ligne par configuration, dans l'ordre de `CONFIGURATIONS`, avec le nombre de réponses, la réussite à 5 de la recherche (la part des réponses dont un passage de référence est servi, sur les questions qui ont une réponse ; rien pour les configurations sans passages), `juste` et sa marge, `fidele` (la fidélité moyenne), `refus_justes`, `refus_a_tort`, `citations` (les moyennes de `refus_juste`, `refus_a_tort` et `citation_juste`) et `secondes`. Puis, question par question, comparez les configurations qui ne diffèrent que d'un composant : le RAG avec le générateur de base, puis affiné ; le générateur affiné avec l'encodeur de base, puis adapté ; le générateur affiné sans passages face au générateur de base sans passages. Affichez les questions gagnées et perdues, et lisez les réponses aux questions perdues.

Quel composant apporte le plus ? Les écarts dépassent-ils la marge ? Le générateur affiné refuse-t-il davantage, à tort ou à raison ? Sans passages, en sait-il plus de mémoire, et que faudrait-il en conclure ? Que coûte chaque configuration en latence ?

**Documentation utile**

- [`DataFrame.pivot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)
- [`DataFrame.reindex`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html)

In [ ]:
# Les modèles de l'évaluation : les vôtres, ou ceux de la référence, sur GPU.
GENERATEUR_EVALUE, JUGE = MODELE, MODELE_JUGE
if not GPU:
  evaluation = pd.read_json("raft-evaluation.jsonl", lines=True)
  evaluation["answer"] = evaluation.answer.astype(object).where(
    evaluation.answer.notna(), None
  )
  GENERATEUR_EVALUE, JUGE = "Qwen/Qwen3-1.7B", "ministral-3:8b"
print(evaluation.groupby("configuration", sort=False).size().to_dict())

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- la réussite à 5 d'une réponse : `bool(set(ligne.servis) & set(ligne.passage_ids))`, à `NaN` pour une question sans réponse ou une configuration sans passages ; `evaluation.apply(fonction, axis=1)` l'applique à chaque ligne
- `evaluation.groupby("configuration").agg(n=("id", "size"), reussite=("reussite", "mean"), juste=("juste", "mean"), ...)`, puis `.reindex(list(CONFIGURATIONS))` : les configurations dans l'ordre du cours
- `tableau["marge"] = [marge(p, n) for p, n in zip(tableau.juste, tableau.n)]`
- `evaluation.pivot(index="id", columns="configuration", values="juste")` : une colonne de succès par configuration ; `.astype(bool)` avant `comparer`
- `evaluation.set_index(["configuration", "id"]).loc[(nom, i)]` : la réponse d'une configuration à une question

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def reussite(ligne) -> float:
  if ligne.answer is None or not ligne.servis:
    return np.nan
  return float(bool(set(ligne.servis) & set(ligne.passage_ids)))


evaluation["reussite"] = evaluation.apply(reussite, axis=1)
tableau = (
  evaluation.groupby("configuration")
  .agg(
    n=("id", "size"),
    reussite=("reussite", "mean"),
    juste=("juste", ...),
    fidele=("fidelite", "mean"),
    refus_justes=("refus_juste", "mean"),
    refus_a_tort=(..., "mean"),
    citations=("citation_juste", "mean"),
    secondes=("secondes", "mean"),
  )
  .reindex(list(CONFIGURATIONS))
)
tableau.insert(3, "marge", [marge(p, n) for p, n in zip(tableau.juste, tableau.n)])
print(tableau.round(2).to_string())

succes = evaluation.pivot(index="id", columns="configuration", values="juste")
par_cle = evaluation.set_index(["configuration", "id"])
for avant, apres in [
  ("RAG + base", "RAG + affiné"),
  ("RAG + affiné", "RAG + encodeur adapté + affiné"),
  ("base, sans passages", ...),
]:
  communes = succes[[avant, apres]].dropna().astype(bool)
  gagnees, perdues = comparer(communes[avant], communes[...])
  print(f"\n{avant} -> {apres} : gagnées {gagnees}, perdues {perdues}")
  for i in perdues[:3]:
    print(f"  Q{i} {par_cle.loc[(apres, i), 'question']}")
    print(f"    avant : {par_cle.loc[(avant, i), 'reponse']}")
    print(f"    après : {par_cle.loc[(apres, i), 'reponse']}")
```

### Solution

In [ ]:
def reussite(ligne) -> float:
  if ligne.answer is None or not ligne.servis:
    return np.nan
  return float(bool(set(ligne.servis) & set(ligne.passage_ids)))


evaluation["reussite"] = evaluation.apply(reussite, axis=1)
tableau = (
  evaluation.groupby("configuration")
  .agg(
    n=("id", "size"),
    reussite=("reussite", "mean"),
    juste=("juste", "mean"),
    fidele=("fidelite", "mean"),
    refus_justes=("refus_juste", "mean"),
    refus_a_tort=("refus_a_tort", "mean"),
    citations=("citation_juste", "mean"),
    secondes=("secondes", "mean"),
  )
  .reindex(list(CONFIGURATIONS))
)
tableau.insert(3, "marge", [marge(p, n) for p, n in zip(tableau.juste, tableau.n)])
print(tableau.round(2).to_string())

succes = evaluation.pivot(index="id", columns="configuration", values="juste")
par_cle = evaluation.set_index(["configuration", "id"])
for avant, apres in [
  ("RAG + base", "RAG + affiné"),
  ("RAG + affiné", "RAG + encodeur adapté + affiné"),
  ("base, sans passages", "affiné, sans passages"),
]:
  communes = succes[[avant, apres]].dropna().astype(bool)
  gagnees, perdues = comparer(communes[avant], communes[apres])
  print(f"\n{avant} -> {apres} : gagnées {gagnees}, perdues {perdues}")
  for i in perdues[:3]:
    print(f"  Q{i} {par_cle.loc[(apres, i), 'question']}")
    print(f"    avant : {par_cle.loc[(avant, i), 'reponse']}")
    print(f"    après : {par_cle.loc[(apres, i), 'reponse']}")

# Chaque comparaison ne change qu'un composant : le générateur affiné face au générateur
# de base, sur les mêmes passages ; l'encodeur adapté face à celui de base, avec le même
# générateur. Un écart plus petit que la marge (près de 13 points sur 60 questions
# autour de 50 %) ne départage rien ; les questions gagnées et perdues disent s'il vient
# de quelques questions ou de toutes. Les refus se lisent ensemble : des refus justes
# qui montent avec des refus à tort qui montent aussi font un modèle qui refuse plus,
# pas un modèle qui lit mieux. Sans passages, le générateur affiné ne devrait pas en
# savoir plus que celui de base : aucun passage de test n'est entré dans l'entraînement,
# et s'il en sait plus, c'est une fuite à chercher. La latence suit la longueur du
# prompt : les passages, pas l'adaptateur.

## 6. Rédiger la recommandation

Une page, que l'équipe relira à la prochaine décision, selon la grille du cours : le besoin et ses contraintes, les configurations mesurées (le tableau, sa marge, la version du jeu de test), le choix et la mesure qui le justifie, ce qu'on a écarté et pourquoi, les coûts (entraînement, service, maintenance), les risques (questions perdues, refus à tort, dépendance à une recherche), et quand revoir la décision.

Écrivez d'abord `choisir(tableau)`, la règle du choix : parmi les trois configurations avec passages (une réponse sans passage ne se source pas), la plus simple dont le taux `juste` est à moins de la marge du meilleur, la plus simple étant celle qui a le moins de composants entraînés, dans l'ordre de `AVEC_PASSAGES`. Puis écrivez `recommandation.md`, en Markdown, les chiffres tirés de vos mesures : le tableau (`tableau.round(2).to_markdown()`), `bilan` (exercice 2), `minutes_encodeur` et `minutes_reindexation` (exercice 3), les questions gagnées et perdues (exercice 5) ; et, en quelques phrases, ce que les chiffres ne disent pas : le corpus de 1914 ne change plus, des questions de lecteurs d'aujourd'hui, un T4 pour l'entraînement.

Relisez votre page : la règle a-t-elle choisi ce que vous auriez choisi ? Un gain de quelques points vaut-il un modèle de plus à maintenir ? Sans GPU, la page porte sur l'évaluation de référence, mais les coûts sont ceux de votre exécution, extrapolés, ce qu'elle doit dire.

**Documentation utile**

- [`DataFrame.to_markdown`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_markdown.html)

In [ ]:
# Du plus simple au plus complexe : aucun composant entraîné, le générateur, les deux.
AVEC_PASSAGES = ["RAG + base", "RAG + affiné", "RAG + encodeur adapté + affiné"]
EMPREINTE = hashlib.sha256(
  jeu.to_json(orient="records", force_ascii=False).encode()
).hexdigest()[:12]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tableau.loc[AVEC_PASSAGES]` : les trois configurations ; `idxmax()` sur leur colonne `juste`, la meilleure ; `next(nom for nom in AVEC_PASSAGES if ...)`, la première qui passe le seuil
- une f-string sur plusieurs lignes (`f"""..."""`) pour la page ; `open("recommandation.md", "w", encoding="utf-8").write(page)`
- `"sur GPU" if GPU else "sans GPU, extrapolé"` : d'où viennent les coûts ; sur 557 exemples, `bilan["minutes"] * 557 / len(entrainement)` extrapole l'entraînement ; de même, `minutes_encodeur * 557 / len(triplets)` celui de l'encodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def choisir(tableau: pd.DataFrame) -> str:
  candidates = tableau.loc[AVEC_PASSAGES]
  meilleure = candidates.loc[candidates.juste.idxmax()]
  seuil = meilleure.juste - meilleure.marge
  return next(nom for nom in AVEC_PASSAGES if candidates.juste[nom] >= ...)


choisie = choisir(tableau)
ecartees = [nom for nom in AVEC_PASSAGES if nom != choisie]
appareil = "sur GPU" if GPU else "sans GPU, extrapolés"
minutes_generateur = bilan["minutes"] * 557 / len(entrainement)
page = f"""# Recommandation : l'assistant de la presse de 1914

## Le besoin
...

## Les configurations mesurées
Jeu de test `{EMPREINTE}` ({len(jeu)} questions, chapitre 4), juge `{MODELE_JUGE}`.

{tableau.round(2).to_markdown()}

## Le choix
**{choisie}** : ...

## Les coûts ({appareil})
- Générateur : {minutes_generateur:.0f} min d'entraînement, ...
- Encodeur : ...

## Les risques
...

## Quand revoir la décision
...
"""
with open("recommandation.md", "w", encoding="utf-8") as fichier:
  fichier.write(page)
print(page)
```

### Solution

In [ ]:
def choisir(tableau: pd.DataFrame) -> str:
  candidates = tableau.loc[AVEC_PASSAGES]
  meilleure = candidates.loc[candidates.juste.idxmax()]
  seuil = meilleure.juste - meilleure.marge
  return next(nom for nom in AVEC_PASSAGES if candidates.juste[nom] >= seuil)


def pourcent(p: float) -> str:
  return f"{p * 100:.0f}\u00a0%"


def duree(minutes: float) -> str:
  return f"{minutes:.0f}\u00a0min" if minutes < 120 else f"{minutes / 60:.0f}\u00a0h"


choisie = choisir(tableau)
ecartees = [nom for nom in AVEC_PASSAGES if nom != choisie]
ligne_choisie = tableau.loc[choisie]
appareil = "sur GPU" if GPU else "sans GPU, extrapolés"
minutes_generateur = bilan["minutes"] * 557 / len(entrainement)
# Sans GPU, l'encodeur n'a vu que quelques triplets : sa durée aussi s'extrapole.
minutes_encodeur_total = minutes_encodeur * (1 if GPU else 557 / len(triplets))
communes = succes[AVEC_PASSAGES].dropna().astype(bool)
_, perdues_choisie = comparer(communes["RAG + base"], communes[choisie])
page = f"""# Recommandation : l'assistant de la presse de 1914

## Le besoin

Répondre aux questions de lecteurs d'aujourd'hui sur la presse française de 1914
(64 598 passages de L'Humanité et du Figaro), une réponse sourcée ou un refus. Le corpus
ne change plus : pas de nouveaux documents à indexer. Données publiques, pas de contrainte
de confidentialité. Un petit modèle servi sur un GPU modeste, un T4 pour l'entraînement ;
une réponse en quelques secondes.

## Les configurations mesurées

Jeu de test `{EMPREINTE}` du chapitre 4 ({len(jeu)} questions, dont
{int(jeu.answer.isna().sum())} sans réponse), générateur `{GENERATEUR_EVALUE}`, recherche
hybride à 5 passages, juge `{JUGE}` (Ragas). La marge de `juste` est à 95 %.

{tableau.round(2).to_markdown()}

## Le choix

**{choisie}** : juste à {pourcent(ligne_choisie.juste)} (± {pourcent(ligne_choisie.marge)}), la plus
simple des configurations avec passages à moins de la marge de la meilleure. Écartées :
{", ".join(ecartees)}, qui ne font pas mieux à la marge près, ou font moins bien. Les
configurations sans passages ne se sourcent pas : elles mesurent ce que le modèle sait de
mémoire.

## Les coûts ({appareil})

- Générateur : {duree(minutes_generateur)} d'entraînement pour 557 exemples ; un
  adaptateur de {bilan["adaptateur_mo"]:.0f} Mo, fusionné : aucun coût de plus en service.
- Encodeur : {duree(minutes_encodeur_total)} d'entraînement sur les paires du split `train`,
  {duree(minutes_reindexation)} de réindexation, à refaire à chaque nouvel encodeur.
- Maintenance : un nouveau modèle de base, une nouvelle consigne ou une autre recherche
  demandent de régénérer le jeu RAFT, de réentraîner et de remesurer.

## Les risques

- Questions perdues face au RAG de base : {perdues_choisie or "aucune"}.
- Refus à tort : {pourcent(ligne_choisie.refus_a_tort)} des réponses dont un passage de
  référence est servi ; refus justes : {pourcent(ligne_choisie.refus_justes)}.
- Le générateur affiné a appris avec les distracteurs de la recherche hybride de base :
  changer de recherche, c'est remesurer la combinaison.
- Le juge n'est contrôlé que sur un échantillon (chapitre 4).

## Quand revoir la décision

- Un nouveau modèle de base, ou un générateur plus grand qui lirait aussi bien sans
  affinage.
- Des questions d'un autre genre que celles du jeu (autre période, autres journaux).
- Une baisse de `juste` au-delà de la marge au test de non-régression.
"""
with open("recommandation.md", "w", encoding="utf-8") as fichier:
  fichier.write(page)
print(page)

# La règle choisit sur la mesure, puis sur le coût ; elle ne pèse pas les questions :
# une question perdue sur un sujet sensible compte plus qu'une gagnée, d'où leur liste
# dans la page. Sans GPU, le tableau est celui de la référence, et les coûts sont
# extrapolés de deux pas et de quatre passages sur deux processeurs : la page le dit.

## À retenir

- RAFT affine le générateur dans les conditions du service : la même consigne, le même format, les mêmes $k$ passages, ceux que la recherche du service classe en tête, l'oracle à une position tirée au hasard. Sans oracle, la cible est un refus : le modèle apprend à vérifier que les passages répondent, pas des faits.
- Les distracteurs et les négatifs se choisissent avec soin : jamais un passage de test, jamais un passage qui donne la réponse (faux distracteur, faux négatif).
- Un adaptateur LoRA sur l'encodeur se garde en quelques dizaines de Mo ; l'index qu'il a réencodé va avec lui, et la réindexation est le premier coût d'un encodeur adapté.
- On compare des configurations qui ne diffèrent que d'un composant, sur le même jeu, avec le même juge, question par question et à la marge près ; les lignes sans passages disent ce que le modèle a appris de mémoire.
- La recommandation choisit sur la mesure, puis sur le coût : à résultats proches, le moins de composants à entraîner et à maintenir.